# Demo prediksi live Final64

Jalankan setelah tersedia 12 snapshot per jam yang berurutan dan lengkap. Input seluruh 64 koridor memakai scaler training; pilih koridor hanya untuk tampilan. Snapshot live bukan agregasi rata-rata per jam. Hasil ini demonstrasi inferensi; akurasi harus diuji dengan observasi masa depan yang sesuai.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd().resolve()
while not (root / "streamlit_app").exists():
    if root.parent == root:
        raise RuntimeError("Buka notebook dari folder proyek")
    root = root.parent
sys.path.insert(0, str(root))
from scripts.live.build_live_latest_12h import build_latest_12h
from streamlit_app.data_loader import load_scaler
from streamlit_app.model_loader import load_graphwavenet_model

X, stations, origin = build_latest_12h()
print("Input:", X.shape, "Origin UTC:", origin)

In [ ]:
mode = "adaptive"  # fixed / adaptive / hybrid
station = stations[0]  # ganti dengan station_id koridor
model = load_graphwavenet_model(mode)
scaler = load_scaler()
prediction = model.predict(X, verbose=0)[0] * scaler["scale"] + scaler["mean"]
idx = stations.index(station)
result = pd.DataFrame(
    {
        "station_id": station,
        "horizon_jam": [1, 3, 6],
        "target_WIB": [
            (origin + pd.Timedelta(hours=h)).tz_convert("Asia/Jakarta") for h in [1, 3, 6]
        ],
        "prediksi_km_jam": prediction[:, idx],
    }
)
display(result)
result.to_csv(root / "data/collected/live_prediction_selected.csv", index=False)